# ***Geopulse:Hyper-Local Mobility Analytics***

In [170]:
import uuid
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

In [171]:
# Set seed for reproducibility
np.random.seed(42)
random.seed(42)

In [172]:
print("Loading, merging, and populating GeoPulse datasets...")

# 1. Generate sample data frames representing the 3 sources
NUM_PINGS = 1000
BASE_LAT, BASE_LON = 40.748817, -73.985428

device_ids = [f"dev_{str(uuid.uuid4())[:8]}" for _ in range(50)]
df_pings = pd.DataFrame([{
    "ping_id": f"px_{str(uuid.uuid4())[:8]}",
    "device_id": random.choice(device_ids),
    "timestamp": datetime(2026, 11, 18, random.randint(6, 22), random.randint(0, 59)),
    "latitude": round(BASE_LAT + np.random.normal(0, 0.012), 6),
    "longitude": round(BASE_LON + np.random.normal(0, 0.012), 6),
    "horizontal_accuracy": round(random.uniform(2.0, 15.0), 1)
} for _ in range(NUM_PINGS)])

df_stores = pd.DataFrame([
    {
        "store_id": "store_coffee_01",
        "store_name": "Store A - 5th Ave",
        "store_status": "EXISTING",
        "store_lat": 40.748500,
        "store_lon": -73.985700
    },
    {
        "store_id": "store_coffee_02",
        "store_name": "Store B - Proposed 6th Ave",
        "store_status": "PROPOSED",
        "store_lat": 40.751000,
        "store_lon": -73.988000
    }
])

df_zones = pd.DataFrame([
    {
        "zone_id": "zone_midtown_01",
        "zone_name": "Midtown South Commercial",
        "zone_type": "High-Density Commercial",
        "base_footfall_category": "Transit Hub & Office Corridor"
    },
    {
        "zone_id": "zone_flatiron_02",
        "zone_name": "Flatiron Retail District",
        "zone_type": "Mixed-Use Commercial",
        "base_footfall_category": "Shopping District"
    }
])



Loading, merging, and populating GeoPulse datasets...


In [173]:
# 2. Perform Spatial Enrichment & Always Assign Nearest Store
merged_rows = []
for _, ping in df_pings.iterrows():
    # Assign zoning attribute based on geographic coordinate threshold
    zone = df_zones.iloc[0].to_dict() if ping['latitude'] > 40.750 else df_zones.iloc[1].to_dict()

    # Always find the nearest store regardless of distance
    closest_store = None
    min_dist = float('inf')
    for _, store in df_stores.iterrows():
        dist = np.sqrt((ping['latitude'] - store['store_lat'])**2 + (ping['longitude'] - store['store_lon'])**2)
        if dist < min_dist:
            min_dist = dist
            closest_store = store

    distance_meters = round(min_dist * 111320, 1)
    is_in_catchment = min_dist <= 0.0045 # ~500m walking radius threshold

    merged_rows.append({
        "ping_id": ping['ping_id'],
        "device_id": ping['device_id'],
        "timestamp": ping['timestamp'],
        "latitude": ping['latitude'],
        "longitude": ping['longitude'],
        "horizontal_accuracy": ping['horizontal_accuracy'],
        "zone_id": zone['zone_id'],
        "zone_name": zone['zone_name'],
        "zone_type": zone['zone_type'],
        "base_footfall_category": zone['base_footfall_category'],
        "nearest_store_id": closest_store['store_id'],         # Populated for all 1000 rows
        "nearest_store_name": closest_store['store_name'],     # Populated for all 1000 rows
        "nearest_store_status": closest_store['store_status'], # Populated for all 1000 rows
        "distance_to_store_meters": distance_meters,
        "is_within_catchment": is_in_catchment
    })

In [174]:
# 3. Create Master DataFrame & Export
df_master = pd.DataFrame(merged_rows)
df_master.to_csv("geo_pulse_master_dataset.csv", index=False)

print("Merge and population completed!")
print(f" - Master dataset generated successfully with {len(df_master)} rows and {len(df_master.columns)} columns.")
print(f" - Saved to `geo_pulse_master_dataset.csv`")

Merge and population completed!
 - Master dataset generated successfully with 1000 rows and 15 columns.
 - Saved to `geo_pulse_master_dataset.csv`


*Saved the Dataset Successfully.*

In [175]:
# Preview the first 10 rows
print(df_master.head(10))

       ping_id     device_id           timestamp   latitude  longitude  \
0  px_637d167d  dev_638f243a 2026-11-18 09:01:00  40.754778 -73.987087   
1  px_621d5aa5  dev_dbdf0e1e 2026-11-18 13:08:00  40.756589 -73.967152   
2  px_f3f2d352  dev_090ea78c 2026-11-18 08:37:00  40.746007 -73.988238   
3  px_404cedd5  dev_361d2051 2026-11-18 08:13:00  40.767768 -73.976219   
4  px_9f905f4d  dev_935bf7d9 2026-11-18 06:35:00  40.743183 -73.978917   
5  px_2650fc7f  dev_789ea21a 2026-11-18 19:14:00  40.743256 -73.991017   
6  px_c4400fce  dev_c836615a 2026-11-18 06:48:00  40.751721 -74.008387   
7  px_af180fee  dev_ecab8ed0 2026-11-18 19:21:00  40.728118 -73.992175   
8  px_d1a21c20  dev_7523c9ab 2026-11-18 16:06:00  40.736663 -73.981657   
9  px_dbed10c6  dev_43af9678 2026-11-18 17:54:00  40.737921 -74.002376   

   horizontal_accuracy           zone_id                 zone_name  \
0                 11.6   zone_midtown_01  Midtown South Commercial   
1                 11.6   zone_midtown_01  Mid

In [176]:
# Preview the last 10 rows
print(df_master.tail(10))

         ping_id     device_id           timestamp   latitude  longitude  \
990  px_c5435db8  dev_c40b1706 2026-11-18 11:10:00  40.737157 -74.001983   
991  px_97544307  dev_41928fa6 2026-11-18 10:02:00  40.741296 -73.975079   
992  px_f0e6c0db  dev_090ea78c 2026-11-18 19:12:00  40.760255 -73.979271   
993  px_de79fc72  dev_638f243a 2026-11-18 10:53:00  40.757518 -73.979234   
994  px_2cd7f490  dev_d254a555 2026-11-18 08:05:00  40.741119 -73.980245   
995  px_308e3e77  dev_446285ae 2026-11-18 16:54:00  40.758422 -73.976377   
996  px_a0672875  dev_69ca8b15 2026-11-18 06:50:00  40.763084 -73.976928   
997  px_4a63754c  dev_25574d0e 2026-11-18 22:50:00  40.753034 -73.972586   
998  px_8ebc9a48  dev_5e935436 2026-11-18 10:09:00  40.748499 -73.996010   
999  px_bf2b051b  dev_25574d0e 2026-11-18 09:59:00  40.746860 -73.994367   

     horizontal_accuracy           zone_id                 zone_name  \
990                 14.6  zone_flatiron_02  Flatiron Retail District   
991                

### ***Loading the Dataset***

In [177]:
# Load the merged master dataset
df_master = pd.read_csv("geo_pulse_master_dataset.csv")
initial_rows = len(df_master)
print(f"Loaded {initial_rows} rows and {len(df_master.columns)} columns.")

Loaded 1000 rows and 15 columns.


In [178]:
# Preview the output
display(df_master.head(15))

,ping_id,device_id,timestamp,latitude,longitude,horizontal_accuracy,zone_id,zone_name,zone_type,base_footfall_category,nearest_store_id,nearest_store_name,nearest_store_status,distance_to_store_meters,is_within_catchment
0,px_637d167d,dev_638f243a,2026-11-18 09:01:00,40.754778,-73.987087,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,432.7,True
1,px_621d5aa5,dev_dbdf0e1e,2026-11-18 13:08:00,40.756589,-73.967152,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,2252.6,False
2,px_f3f2d352,dev_090ea78c,2026-11-18 08:37:00,40.746007,-73.988238,7.5,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,396.0,True
3,px_404cedd5,dev_361d2051,2026-11-18 08:13:00,40.767768,-73.976219,5.0,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,2281.3,False
4,px_9f905f4d,dev_935bf7d9,2026-11-18 06:35:00,40.743183,-73.978917,4.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,959.4,False
5,px_2650fc7f,dev_789ea21a,2026-11-18 19:14:00,40.743256,-73.991017,7.8,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,831.3,False
6,px_c4400fce,dev_c836615a,2026-11-18 06:48:00,40.751721,-74.008387,12.5,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,2270.9,False
7,px_af180fee,dev_ecab8ed0,2026-11-18 19:21:00,40.728118,-73.992175,5.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,2380.7,False
8,px_d1a21c20,dev_7523c9ab,2026-11-18 16:06:00,40.736663,-73.981657,3.2,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,1392.4,False
9,px_dbed10c6,dev_43af9678,2026-11-18 17:54:00,40.737921,-74.002376,6.5,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,2163.5,False


### ***Dataset Description***

In [179]:
df_master.shape

(1000, 15)

*1000 rows and 15 columns*

In [180]:
print(f"Original row count: {len(df_master)}")

Original row count: 1000


### ***Attributes of the dataset***

In [181]:
df_master.columns

Index(['ping_id', 'device_id', 'timestamp', 'latitude', 'longitude',
       'horizontal_accuracy', 'zone_id', 'zone_name', 'zone_type',
       'base_footfall_category', 'nearest_store_id', 'nearest_store_name',
       'nearest_store_status', 'distance_to_store_meters',
       'is_within_catchment'],
      dtype='object')

### ***Checking the consistency of the dataset.***

*First Five Rows*

In [182]:
df_master.head()

,ping_id,device_id,timestamp,latitude,longitude,horizontal_accuracy,zone_id,zone_name,zone_type,base_footfall_category,nearest_store_id,nearest_store_name,nearest_store_status,distance_to_store_meters,is_within_catchment
0,px_637d167d,dev_638f243a,2026-11-18 09:01:00,40.754778,-73.987087,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,432.7,True
1,px_621d5aa5,dev_dbdf0e1e,2026-11-18 13:08:00,40.756589,-73.967152,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,2252.6,False
2,px_f3f2d352,dev_090ea78c,2026-11-18 08:37:00,40.746007,-73.988238,7.5,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,396.0,True
3,px_404cedd5,dev_361d2051,2026-11-18 08:13:00,40.767768,-73.976219,5.0,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,2281.3,False
4,px_9f905f4d,dev_935bf7d9,2026-11-18 06:35:00,40.743183,-73.978917,4.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,959.4,False


*Last Five Rows*

In [183]:
df_master.tail()

,ping_id,device_id,timestamp,latitude,longitude,horizontal_accuracy,zone_id,zone_name,zone_type,base_footfall_category,nearest_store_id,nearest_store_name,nearest_store_status,distance_to_store_meters,is_within_catchment
995,px_308e3e77,dev_446285ae,2026-11-18 16:54:00,40.758422,-73.976377,5.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,1515.6,False
996,px_a0672875,dev_69ca8b15,2026-11-18 06:50:00,40.763084,-73.976928,11.1,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,1824.5,False
997,px_4a63754c,dev_25574d0e,2026-11-18 22:50:00,40.753034,-73.972586,7.5,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,1544.6,False
998,px_8ebc9a48,dev_5e935436,2026-11-18 10:09:00,40.748499,-73.996010,9.4,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,934.1,False
999,px_bf2b051b,dev_25574d0e,2026-11-18 09:59:00,40.746860,-73.994367,3.4,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,845.4,False


### ***Overview of the Dataset***

In [184]:
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ping_id                   1000 non-null   object 
 1   device_id                 1000 non-null   object 
 2   timestamp                 1000 non-null   object 
 3   latitude                  1000 non-null   float64
 4   longitude                 1000 non-null   float64
 5   horizontal_accuracy       1000 non-null   float64
 6   zone_id                   1000 non-null   object 
 7   zone_name                 1000 non-null   object 
 8   zone_type                 1000 non-null   object 
 9   base_footfall_category    1000 non-null   object 
 10  nearest_store_id          1000 non-null   object 
 11  nearest_store_name        1000 non-null   object 
 12  nearest_store_status      1000 non-null   object 
 13  distance_to_store_meters  1000 non-null   float64
 14  is_within

### ***Summary Statistics of the Numerical Features***

In [185]:
df_master.describe()

,latitude,longitude,horizontal_accuracy,distance_to_store_meters
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,40.749215,-73.984744,8.499400,1551.998900
std,0.011539,0.012180,3.753861,877.892088
min,40.709922,-74.020713,2.000000,43.100000
25%,40.741478,-73.993245,5.300000,893.650000
50%,40.749250,-73.984855,8.400000,1425.650000
75%,40.756597,-73.976849,11.600000,2111.525000
max,40.785764,-73.939195,15.000000,5227.200000


## ***Data Preprocessing***

### ***Data Cleaning Phase***

***1.Removing Duplicate Values***

In [186]:
# 1. Remove duplicate pings and parse timestamps
df_master = df_master.drop_duplicates(subset=["ping_id"]).copy()
df_master["timestamp"] = pd.to_datetime(df_master["timestamp"], errors="coerce")
# Drop rows missing critical identifiers, timestamps, or coordinates
df_master = df_master.dropna(subset=["ping_id", "device_id", "timestamp", "latitude", "longitude"])

***2.Checking for Null Values***

In [187]:
df_master.isnull().sum()

,0
ping_id,0
device_id,0
timestamp,0
latitude,0
longitude,0
horizontal_accuracy,0
zone_id,0
zone_name,0
zone_type,0
base_footfall_category,0


*Output: No Null Values.*

 ***3.Datatype of Each Columns***

In [188]:
df_master.dtypes

,0
ping_id,object
device_id,object
timestamp,datetime64[ns]
latitude,float64
longitude,float64
horizontal_accuracy,float64
zone_id,object
zone_name,object
zone_type,object
base_footfall_category,object


In [189]:
# Ensure timestamp is parsed as datetime
df_master["timestamp"] = pd.to_datetime(df_master["timestamp"], errors="coerce")

# High-cardinality identifiers as string dtype
id_cols = ["ping_id", "device_id"]
for col in id_cols:
    if col in df_master.columns:
        df_master[col] = df_master[col].astype("string")

# Low-cardinality attributes as category dtype (great for grouping & memory)
category_cols = [
    "zone_id",
    "zone_name",
    "zone_type",
    "base_footfall_category",
    "nearest_store_id",
    "nearest_store_name",
    "nearest_store_status",
]
for col in category_cols:
    if col in df_master.columns:
        df_master[col] = df_master[col].astype("category")

# Numeric & boolean columns
float_cols = ["latitude", "longitude", "horizontal_accuracy", "distance_to_store_meters"]
for col in float_cols:
    if col in df_master.columns:
        df_master[col] = pd.to_numeric(df_master[col], errors="coerce").astype("float64")

if "is_within_catchment" in df_master.columns:
    df_master["is_within_catchment"] = df_master["is_within_catchment"].astype("bool")

# Verify updated dtypes
df_master.dtypes

,0
ping_id,string[python]
device_id,string[python]
timestamp,datetime64[ns]
latitude,float64
longitude,float64
horizontal_accuracy,float64
zone_id,category
zone_name,category
zone_type,category
base_footfall_category,category


*Output: Converted timestamp to "datetime64[ns]", used "string" for identifiers and "category" for low-cardinality spatial/store labels makes filtering, grouping, and time-based aggregations in pandas cleaner and faster.*

***4.Validating the Geographic Bounds and GPS Precision***

In [190]:
# 4. Validate geographic bounds and GPS precision
df_master["latitude"] = pd.to_numeric(df_master["latitude"], errors="coerce")
df_master["longitude"] = pd.to_numeric(df_master["longitude"], errors="coerce")
df_master["horizontal_accuracy"] = pd.to_numeric(df_master["horizontal_accuracy"], errors="coerce")
df_master["distance_to_store_meters"] = pd.to_numeric(df_master["distance_to_store_meters"], errors="coerce")

df_master = df_master[
    (df_master["latitude"].between(-90.0, 90.0)) &
    (df_master["longitude"].between(-180.0, 180.0))
]

# Filter out low-precision GPS pings (e.g., accuracy worse than 20 meters)
df_master = df_master[df_master["horizontal_accuracy"].fillna(0) <= 20.0]

# Ensure non-negative distance values
df_master = df_master.dropna(subset=["distance_to_store_meters"])
df_master["distance_to_store_meters"] = df_master["distance_to_store_meters"].abs()
display(df_master.head())

,ping_id,device_id,timestamp,latitude,longitude,horizontal_accuracy,zone_id,zone_name,zone_type,base_footfall_category,nearest_store_id,nearest_store_name,nearest_store_status,distance_to_store_meters,is_within_catchment
0,px_637d167d,dev_638f243a,2026-11-18 09:01:00,40.754778,-73.987087,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,432.7,True
1,px_621d5aa5,dev_dbdf0e1e,2026-11-18 13:08:00,40.756589,-73.967152,11.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,2252.6,False
2,px_f3f2d352,dev_090ea78c,2026-11-18 08:37:00,40.746007,-73.988238,7.5,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,396.0,True
3,px_404cedd5,dev_361d2051,2026-11-18 08:13:00,40.767768,-73.976219,5.0,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,2281.3,False
4,px_9f905f4d,dev_935bf7d9,2026-11-18 06:35:00,40.743183,-73.978917,4.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,959.4,False


***5.Standardizing String Fields and Boolean Catchment Flag***

In [191]:
# 5. Standardize string fields and boolean catchment flag
text_cols = [
    "ping_id", "device_id", "zone_id", "zone_name",
    "zone_type", "base_footfall_category", "nearest_store_id",
    "nearest_store_name", "nearest_store_status"
]
for col in text_cols:
    if col in df_master.columns:
        df_master[col] = df_master[col].astype(str).str.strip()

df_master["nearest_store_status"] = df_master["nearest_store_status"].str.upper()

# Recalculate catchment flag consistently using a 500m radius
df_master["is_within_catchment"] = df_master["distance_to_store_meters"] <= 500.0

print("--- Sample of Standardized Columns ---")
cols_to_show = [
    "ping_id", "nearest_store_name", "nearest_store_status",
    "distance_to_store_meters", "is_within_catchment"
]
print(df_master[cols_to_show].head(10).to_string(index=False))

print("\n--- Store Status Counts ---")
print(df_master["nearest_store_status"].value_counts(dropna=False))

print("\n--- Catchment Flag Counts (<= 500m) ---")
print(df_master["is_within_catchment"].value_counts(dropna=False))

# Check if any string columns still have leading or trailing whitespace
whitespace_issues = {
    col: int((df_master[col] != df_master[col].str.strip()).sum())
    for col in text_cols
    if col in df_master.columns
}
print("\n--- Remaining Whitespace Counts ---")
print(whitespace_issues)

--- Sample of Standardized Columns ---
    ping_id         nearest_store_name nearest_store_status  distance_to_store_meters  is_within_catchment
px_637d167d Store B - Proposed 6th Ave             PROPOSED                     432.7                 True
px_621d5aa5          Store A - 5th Ave             EXISTING                    2252.6                False
px_f3f2d352          Store A - 5th Ave             EXISTING                     396.0                 True
px_404cedd5 Store B - Proposed 6th Ave             PROPOSED                    2281.3                False
px_9f905f4d          Store A - 5th Ave             EXISTING                     959.4                False
px_2650fc7f          Store A - 5th Ave             EXISTING                     831.3                False
px_c4400fce Store B - Proposed 6th Ave             PROPOSED                    2270.9                False
px_af180fee          Store A - 5th Ave             EXISTING                    2380.7                Fals

***Extracting Temporal and Commuter Window Features: Separate Columns for Date and Time derived from the Timestamp Column***

In [192]:
# 6. Preprocessing: Extract temporal and commuter window features
df_master["event_date"] = df_master["timestamp"].dt.date
df_master["event_hour"] = df_master["timestamp"].dt.hour
df_master["hour_bucket"] = df_master["timestamp"].dt.floor("h")

df_master["time_period"] = np.select(
    [
        df_master["event_hour"].between(6, 10),
        df_master["event_hour"].between(11, 16),
        df_master["event_hour"].between(17, 21),
    ],
    ["Morning Commute", "Midday", "Evening Rush"],
    default="Off-Peak",
)

print("--- Sample of Temporal & Commuter Features ---")
cols_to_show = [
    "ping_id", "timestamp", "event_date",
    "event_hour", "hour_bucket", "time_period"
]
print(df_master[cols_to_show].head(10).to_string(index=False))

print("\n--- Pings by Commuter Window ---")
print(df_master["time_period"].value_counts(dropna=False))

print("\n--- Hourly Distribution (Sorted by Hour) ---")
print(df_master["event_hour"].value_counts().sort_index())

--- Sample of Temporal & Commuter Features ---
    ping_id           timestamp event_date  event_hour         hour_bucket     time_period
px_637d167d 2026-11-18 09:01:00 2026-11-18           9 2026-11-18 09:00:00 Morning Commute
px_621d5aa5 2026-11-18 13:08:00 2026-11-18          13 2026-11-18 13:00:00          Midday
px_f3f2d352 2026-11-18 08:37:00 2026-11-18           8 2026-11-18 08:00:00 Morning Commute
px_404cedd5 2026-11-18 08:13:00 2026-11-18           8 2026-11-18 08:00:00 Morning Commute
px_9f905f4d 2026-11-18 06:35:00 2026-11-18           6 2026-11-18 06:00:00 Morning Commute
px_2650fc7f 2026-11-18 19:14:00 2026-11-18          19 2026-11-18 19:00:00    Evening Rush
px_c4400fce 2026-11-18 06:48:00 2026-11-18           6 2026-11-18 06:00:00 Morning Commute
px_af180fee 2026-11-18 19:21:00 2026-11-18          19 2026-11-18 19:00:00    Evening Rush
px_d1a21c20 2026-11-18 16:06:00 2026-11-18          16 2026-11-18 16:00:00          Midday
px_dbed10c6 2026-11-18 17:54:00 2026-11-18 

***7.Saved the Cleaned Dataset***

In [193]:
# 6. Sort chronologically and save cleaned dataset
df_master = df_master.sort_values(["device_id", "timestamp"]).reset_index(drop=True)
df_master.to_csv("geo_pulse_cleaned_dataset.csv", index=False)

cleaned_rows = len(df_master)
print("Preprocessing and cleaning complete!")
print(f" - Rows before cleaning: {initial_rows}")
print(f" - Rows after cleaning:  {cleaned_rows} ({initial_rows - cleaned_rows} removed)")
print(f" - Missing values remaining: {int(df_master.isnull().sum().sum())}")
print(" - Saved to `geo_pulse_cleaned_dataset.csv`")

Preprocessing and cleaning complete!
 - Rows before cleaning: 1000
 - Rows after cleaning:  1000 (0 removed)
 - Missing values remaining: 0
 - Saved to `geo_pulse_cleaned_dataset.csv`


In [194]:
# Preview the output
display(df_master.head(15))

,ping_id,device_id,timestamp,latitude,longitude,horizontal_accuracy,zone_id,zone_name,zone_type,base_footfall_category,nearest_store_id,nearest_store_name,nearest_store_status,distance_to_store_meters,is_within_catchment,event_date,event_hour,hour_bucket,time_period
0,px_4209f73a,dev_03cddc65,2026-11-18 06:05:00,40.744468,-73.998864,14.3,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,1411.1,False,2026-11-18,6,2026-11-18 06:00:00,Morning Commute
1,px_23e144a4,dev_03cddc65,2026-11-18 06:40:00,40.763295,-73.995231,8.7,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,1587.8,False,2026-11-18,6,2026-11-18 06:00:00,Morning Commute
2,px_0230c89b,dev_03cddc65,2026-11-18 08:30:00,40.756187,-73.976338,6.5,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,1348.5,False,2026-11-18,8,2026-11-18 08:00:00,Morning Commute
3,px_0f750060,dev_03cddc65,2026-11-18 08:33:00,40.737973,-73.981536,13.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,1260.2,False,2026-11-18,8,2026-11-18 08:00:00,Morning Commute
4,px_5e7d9ad7,dev_03cddc65,2026-11-18 08:36:00,40.752805,-73.994410,10.2,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,741.3,False,2026-11-18,8,2026-11-18 08:00:00,Morning Commute
5,px_9d220ec0,dev_03cddc65,2026-11-18 09:06:00,40.771401,-73.969283,5.1,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,3082.0,False,2026-11-18,9,2026-11-18 09:00:00,Morning Commute
6,px_db6dd7ea,dev_03cddc65,2026-11-18 10:14:00,40.753550,-73.990480,8.0,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_02,Store B - Proposed 6th Ave,PROPOSED,396.0,True,2026-11-18,10,2026-11-18 10:00:00,Morning Commute
7,px_eb6eb5ec,dev_03cddc65,2026-11-18 10:21:00,40.734490,-73.983713,11.7,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,1575.2,False,2026-11-18,10,2026-11-18 10:00:00,Morning Commute
8,px_4ff59e4a,dev_03cddc65,2026-11-18 10:54:00,40.750976,-73.968724,8.6,zone_midtown_01,Midtown South Commercial,High-Density Commercial,Transit Hub & Office Corridor,store_coffee_01,Store A - 5th Ave,EXISTING,1909.8,False,2026-11-18,10,2026-11-18 10:00:00,Morning Commute
9,px_3a45b651,dev_03cddc65,2026-11-18 11:58:00,40.718170,-73.974216,13.6,zone_flatiron_02,Flatiron Retail District,Mixed-Use Commercial,Shopping District,store_coffee_01,Store A - 5th Ave,EXISTING,3610.3,False,2026-11-18,11,2026-11-18 11:00:00,Midday
